# 07 — Adapter Blending

Interpolate two artist LoRA adapters in **delta space** to produce a continuum of
blended styles, then measure where the blend lands with the attribution classifier.

For a LoRA, the weight update is `ΔW = (α/r)·B·A`. We want

```
ΔW_blend(a) = a·ΔW_gojira + (1−a)·ΔW_tool
```

Averaging `A` and `B` separately is **wrong** (it adds cross-terms `B_g·A_t`).
The exact low-rank form is **rank concatenation** → a rank-`2r` adapter:

```
A_blend = [ a·A_gojira ; (1−a)·A_tool ]     # (2r, in)
B_blend = [ B_gojira , B_tool ]             # (out, 2r)
new config: r → 2r,  lora_alpha → 2·lora_alpha   (keeps scale α/r constant)
```

Endpoints recover the pure adapters (`a=1` zeroes the tool rows), which is the
built-in sanity check.

Building the blended adapters and validating them numerically is **CPU-only**.
Only the generation/eval section at the bottom needs the GPU.

In [ ]:
import json
from pathlib import Path

import torch
from safetensors.torch import load_file, save_file

ADAPTERS = Path("adapters")
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]   # 1 = pure Gojira, 0 = pure Tool
SRC_A, SRC_B = "gojira_lora_r8", "tool_lora_r8"

## 1. Blend builder (CPU)

In [ ]:
def load_lora(name):
    sd = load_file(ADAPTERS / name / "adapter_model.safetensors")
    cfg = json.load(open(ADAPTERS / name / "adapter_config.json"))
    return sd, cfg


def blend_adapters(name_a, name_b, alpha, out_name=None):
    """Delta-space interpolation alpha*A + (1-alpha)*B via rank concatenation.

    Both inputs must be plain LoRA with identical r / lora_alpha / key sets.
    """
    sd_a, cfg_a = load_lora(name_a)
    sd_b, cfg_b = load_lora(name_b)

    assert cfg_a["r"] == cfg_b["r"] and cfg_a["lora_alpha"] == cfg_b["lora_alpha"]
    assert not cfg_a.get("use_dora") and not cfg_b.get("use_dora"), "DoRA not supported"
    assert set(sd_a) == set(sd_b), "adapter key sets differ"

    blended = {}
    for k in sd_a:
        ta, tb = sd_a[k].float(), sd_b[k].float()
        if k.endswith("lora_A.weight"):          # (r, in)  -> concat on rank dim 0
            t = torch.cat([alpha * ta, (1 - alpha) * tb], dim=0)
        elif k.endswith("lora_B.weight"):        # (out, r) -> concat on rank dim 1
            t = torch.cat([ta, tb], dim=1)
        else:
            raise ValueError(f"unexpected key {k}")
        blended[k] = t.to(sd_a[k].dtype)

    out_name = out_name or f"blend_{name_a.split('_')[0]}_{name_b.split('_')[0]}_a{alpha:.2f}"
    out_dir = ADAPTERS / out_name
    out_dir.mkdir(parents=True, exist_ok=True)

    cfg = dict(cfg_a)
    cfg["r"] = cfg_a["r"] * 2
    cfg["lora_alpha"] = cfg_a["lora_alpha"] * 2      # preserve scale = alpha / r
    cfg["inference_mode"] = True
    json.dump(cfg, open(out_dir / "adapter_config.json", "w"), indent=2)
    save_file(blended, out_dir / "adapter_model.safetensors")
    print(f"wrote {out_dir}  (r={cfg['r']}, alpha={cfg['lora_alpha']}, {len(blended)} tensors)")
    return out_name

In [ ]:
blend_names = [blend_adapters(SRC_A, SRC_B, a) for a in ALPHAS]
blend_names

## 2. Numerical validation (CPU)

Materialize `ΔW = (α/r)·B·A` for one layer from the pure adapters and from each
blend, and check `ΔW_blend ≈ a·ΔW_gojira + (1−a)·ΔW_tool` to floating-point
tolerance. No GPU, no base model needed.

In [ ]:
def materialize_delta(name, prefix):
    sd = load_file(ADAPTERS / name / "adapter_model.safetensors")
    cfg = json.load(open(ADAPTERS / name / "adapter_config.json"))
    scale = cfg["lora_alpha"] / cfg["r"]
    A = sd[prefix + ".lora_A.weight"].float()
    B = sd[prefix + ".lora_B.weight"].float()
    return scale * (B @ A)


PREFIX = "base_model.model.model.language_model.layers.0.mlp.gate_proj"
d_g = materialize_delta(SRC_A, PREFIX)
d_t = materialize_delta(SRC_B, PREFIX)

for a, name in zip(ALPHAS, blend_names):
    d_b = materialize_delta(name, PREFIX)
    expected = a * d_g + (1 - a) * d_t
    err = (d_b - expected).abs().max().item()
    print(f"alpha={a:.2f}  max|d_blend - (a*d_g + (1-a)*d_t)| = {err:.2e}")
    assert err < 1e-4, f"blend mismatch at alpha={a}"

print("\nall blends match delta-space interpolation")

## 3. GPU evaluation  *(run when the GPU is free)*

Loads the 4-bit base model, attaches each blended adapter, generates lyrics, and
scores them with the attribution classifier. Endpoints (`a=0`, `a=1`) should match
the pure-adapter numbers from `06_evaluation.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoModelForSequenceClassification,
)
from peft import PeftModel

model_path = "./models/gemma-4-E4B"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    model_path, quantization_config=bnb_config, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

clf_model = AutoModelForSequenceClassification.from_pretrained("./classifier_output/best_model")
clf_tokenizer = AutoTokenizer.from_pretrained("./classifier_output/best_model")
clf_model.eval()
labels = clf_model.config.id2label

In [ ]:
def generate_lyrics(model, prompt="Write song lyrics.\n\n", max_new_tokens=512, min_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        min_new_tokens=min_new_tokens,
        temperature=0.9,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
    )
    return tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)


def classify(text):
    enc = clf_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        logits = clf_model(**enc).logits
        probs = torch.softmax(logits, dim=-1)[0]
    return {labels[j]: probs[j].item() for j in range(len(labels))}


def evaluate_adapter(adapter_path, n_samples=10):
    model = PeftModel.from_pretrained(base_model, str(adapter_path))
    rows, samples = [], []
    for i in range(n_samples):
        text = generate_lyrics(model)
        samples.append(text)
        rows.append(classify(text))
    model.unload()
    return samples, pd.DataFrame(rows)

In [ ]:
N_SAMPLES = 10
records, all_samples = [], {}
for a, name in zip(ALPHAS, blend_names):
    samples, df = evaluate_adapter(ADAPTERS / name, n_samples=N_SAMPLES)
    all_samples[a] = samples
    records.append({
        "alpha": a,
        "Gojira": df["Gojira"].mean(), "Gojira_std": df["Gojira"].std(),
        "Tool": df["Tool"].mean(),     "Tool_std": df["Tool"].std(),
    })
    print(f"alpha={a:.2f}  Gojira={records[-1]['Gojira']:.3f}  Tool={records[-1]['Tool']:.3f}")

blend_df = pd.DataFrame(records)
blend_df

## 4. Interpolation figure

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["pdf.fonttype"] = 42

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(blend_df["alpha"], blend_df["Gojira"], yerr=blend_df["Gojira_std"],
            marker="o", capsize=4, color="#1565c0", label="Gojira attribution")
ax.errorbar(blend_df["alpha"], blend_df["Tool"], yerr=blend_df["Tool_std"],
            marker="s", capsize=4, color="#e65100", label="Tool attribution")
ax.set_xlabel(r"Blend weight $\alpha$   (1 = pure Gojira,  0 = pure Tool)")
ax.set_ylabel("Mean classifier attribution")
ax.set_title("Style Interpolation via Adapter Blending", fontweight="bold")
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig("../report/figures/blending_interpolation.pdf", bbox_inches="tight", dpi=300)
plt.show()